# Montgomery County Traffic Violations: Scope and Source
## tl;dr
This project studies **recorded traffic violations in Montgomery County, Maryland**, not all Maryland drivers or all crashes. Multiple violation records can share a traffic-stop ID. The modeling task is retrospective classification of the recorded `Accident` flag, not predicting an individual's future crash risk.

The completed preparation audit is recorded in `reports/data_audit.json`. Model results and limitations are reported separately in notebooks 4 and 5.

## Context & Methods
The exploratory question is how recorded accident association varies by stop timing and administrative violation disposition. The modeling question is whether recorded stop and vehicle context improves classification over a majority-class baseline.

### Key Assumptions
- Government publication establishes provenance, not representativeness or error-free data.
- The dataset covers issued electronic violations. It has no denominator for traffic exposure, unrecorded behavior, or all drivers.
- `Accident` and `Contributed To Accident` are distinct fields. The model target is `Accident`.
- `Violation Type` contains administrative dispositions such as Citation and Warning; it does not directly encode speeding versus DUI.
- Race and gender are excluded from the revised model. No causal or enforcement-targeting recommendation follows from this project.

Sources: [Data.gov catalog](https://catalog.data.gov/dataset/traffic-violations) and [county publisher](https://data.montgomerycountymd.gov/d/4mse-ku6q). The local snapshot predates the changing online source.

## Data
Download the raw CSV as `Traffic_Violations.csv` in the repository root. See `data/README.md`. All paths resolve locally; Colab upload widgets are unnecessary.

In [1]:
from pathlib import Path
import sys
ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
             if (candidate / "src/traffic_analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the traffic project repository.")
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 15)

Matplotlib is building the font cache; this may take a moment.


In [2]:
raw_path = ROOT / "Traffic_Violations.csv"
if not raw_path.exists():
    raise FileNotFoundError("Download Traffic_Violations.csv using data/README.md before running.")
preview = pd.read_csv(raw_path, nrows=5000, dtype="string")
display(pd.DataFrame({"column": preview.columns, "preview_missing": preview.isna().sum().values}))

,column,preview_missing
0,SeqID,0
1,Date Of Stop,0
2,Time Of Stop,0
3,Agency,0
4,SubAgency,0
...,...,...
38,Driver City,0
39,Driver State,0
40,DL State,0
41,Arrest Type,0


## Results
The table below is a **5,000-row preview**, not a population estimate. Full-source totals are produced by notebook 2. Only a small subset of fields is displayed; raw row-level data is not redistributed in the Git repository.

In [3]:
display(preview[["Date Of Stop", "Time Of Stop", "Accident", "Violation Type", "VehicleType"]].head())
display(preview["Accident"].value_counts(dropna=False).rename("preview_records"))

,Date Of Stop,Time Of Stop,Accident,Violation Type,VehicleType
0,05/01/2023,23:11:00,No,Citation,02 - Automobile
1,11/25/2023,00:20:00,No,Citation,02 - Automobile
2,11/25/2023,00:20:00,No,Citation,02 - Automobile
3,11/26/2023,09:16:00,No,Citation,02 - Automobile
4,11/25/2023,05:45:00,No,Citation,02 - Automobile


Accident
No     4822
Yes     178
Name: preview_records, dtype: Int64

## Takeaways
The record grain and source coverage define what can be concluded. Continue to notebook 2 for reproducible cleaning and exclusion counts; a high classification accuracy alone will not establish useful accident detection.